In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
from datetime import datetime, UTC
import uuid

# use catalog
spark.sql("USE CATALOG novacart_casestudy")

# create schema
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze_schema")

# create ingestion control table
spark.sql("""
CREATE TABLE IF NOT EXISTS novacart_casestudy.bronze_schema.ingestion_control (
    layer STRING,
    table_name STRING,
    ts_col STRING,
    pk_col STRING,
    last_successful_ts TIMESTAMP,
    last_successful_pk BIGINT,
    last_run_id STRING,
    rows_written BIGINT,
    run_status STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

# table config
table_config = {
    "orders": {
        "pk_col": "order_id",
        "ts_col": "updated_at"
    },
    "products": {
        "pk_col": "product_id",
        "ts_col": "updated_at"
    },
    "payments": {
        "pk_col": "payment_id",
        "ts_col": "processed_at"
    }
}

bronze_run_id = str(uuid.uuid4())
print("Current Bronze run id:", bronze_run_id)


def get_last_successful_watermark(table_name: str):
    rows = (
        spark.table("novacart_casestudy.bronze_schema.ingestion_control")
        .filter(
            (f.col("layer") == "bronze") &
            (f.col("table_name") == table_name) &
            (f.col("run_status") == "success")
        )
        .select("last_successful_ts", "last_successful_pk")
        .orderBy(f.col("updated_at").desc())
        .limit(1)
        .collect()
    )

    if not rows:
        return None, None

    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]


def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.now(UTC).replace(tzinfo=None)
        )],
        schema="""
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "novacart_casestudy.bronze_schema.ingestion_control")

    (
        dt.alias("t")
        .merge(
            control_df.alias("s"),
            "t.table_name = s.table_name AND t.layer = s.layer"
        )
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "pk_col": "s.pk_col",
            "last_successful_ts": "s.last_successful_ts",
            "last_successful_pk": "s.last_successful_pk",
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


for table_name, cfg in table_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]

    source_table = f"`novacart_casestudy-sql-connection_catalog`.dbo.{table_name}"
    target_table = f"novacart_casestudy.bronze_schema.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)

    print(f"\n=== Processing {table_name} ===")
    print(f"Last successful ts: {last_successful_ts}")
    print(f"Last successful pk: {last_successful_pk}")

    source_df = (
        spark.read.table(source_table)
        .withColumn(ts_col, f.col(ts_col).cast("timestamp"))
    )

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (f.col(ts_col) > f.lit(last_successful_ts)) |
            (
                (f.col(ts_col) == f.lit(last_successful_ts)) &
                (f.col(pk_col).cast("long") > f.lit(int(last_successful_pk)))
            )
        )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", f.current_timestamp())
        .withColumn("bronze_run_id", f.lit(bronze_run_id))
        .withColumn("bronze_source_table", f.lit(source_table))
    )

    row_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {row_count}")

    if row_count == 0:
        print(f"No new rows for {table_name}.")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            0,
            bronze_run_id
        )
        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    max_ts = (
        rows_to_load
        .agg(f.max(ts_col).alias("max_ts"))
        .collect()[0]["max_ts"]
    )

    max_pk = (
        rows_to_load
        .filter(f.col(ts_col) == f.lit(max_ts))
        .agg(f.max(f.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        row_count,
        bronze_run_id
    )

    print(f"Wrote {row_count} rows to {target_table}")


print("Orders Bronze count:", spark.sql("select count(*) from novacart_casestudy.bronze_schema.orders_raw").collect()[0][0])
print("Payments Bronze count:", spark.sql("select count(*) from novacart_casestudy.bronze_schema.payments_raw").collect()[0][0])
print("Products Bronze count:", spark.sql("select count(*) from novacart_casestudy.bronze_schema.products_raw").collect()[0][0])

display(
    spark.sql("select * from novacart_casestudy.bronze_schema.ingestion_control")
    .orderBy("table_name")
)